In [23]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import seaborn.objects as so

sns.set_theme()

In [ ]:
characterization_files = [
    "data\\u1_xl3d.csv",
    "data\\u2_xl3d.csv",
]
all_frames = []
for data_file in characterization_files:
    data = pd.read_csv(
        data_file,
    )
    all_frames.append(data)

all_data = pd.concat(all_frames)
all_data = all_data.reset_index(drop=True)
display(all_data)

# Scale factor

With 1g +Z reference sensor data, adjust the scaling constant until the scaled box plot converges on 1g

In [ ]:
u1_scale_factor = 48.62e-3
u2_scale_factor = 49.15e-3

def scale_z_by_qtpy_id(row):
    scaling_factor = u1_scale_factor
    if row["QTPy identifier"] == "48ca43576cdc":
        scaling_factor = u2_scale_factor
    return row["Raw Z"] * scaling_factor

all_data["Scaled Z (g)"] = all_data.apply(scale_z_by_qtpy_id, axis="columns")
display(all_data)

In [ ]:
figure = plt.figure(figsize=(18,6))

plot = (
    so.Plot(
        all_data,
        x="Samples averaged",
        y="Raw Z",
        color="QTPy identifier",
    )
    .add(so.Dot(), so.Jitter(x=0.1))
    .scale(x="log")
    .on(figure)
)

plot.show()

In [ ]:
by_qtpy = all_data.groupby(["QTPy identifier"])
display(by_qtpy.size())

In [ ]:
for name, subframe in by_qtpy:
    title = f"Unit {name[0]}"
    boxplot = subframe.plot.box(by="Samples averaged", column="Raw Z", rot=45)
    for axis in boxplot:
        axis.set_title(title)

In [ ]:
std_frames = []
for name, subframe in by_qtpy:
    title = f"Unit {name[0]}"
    std_by_averages = subframe.groupby("Samples averaged")["Raw Z"].std()
    qtpy_xl3d_std = pd.DataFrame({
        "QTPy identifier": name[0],
        "std": std_by_averages,
    })
    std_frames.append(qtpy_xl3d_std)

all_std_frames = pd.concat(std_frames)
display(all_std_frames)

In [ ]:
std_bar_groups = all_std_frames.groupby(["QTPy identifier"])
std_bars_by_qtpy = pd.DataFrame()
for name, subframe in std_bar_groups:
    column_name = f"Unit {name[0]}"
    std_bars_by_qtpy[column_name] = subframe["std"]

display(std_bars_by_qtpy)

In [ ]:
std_bars = std_bars_by_qtpy.plot.bar(figsize=(12,6))
std_bars.set_ylabel("Raw Z std")

In [ ]:
boxen = all_data.groupby("QTPy identifier").plot.box(by="Samples averaged", figsize=(16, 6), rot=45, title="Boxplot by Samples Averaged for ADXL375")